# Cyber Security Attack Type Detection - ML Pipeline

This notebook contains the complete machine learning pipeline for detecting cybersecurity attack types.

## Pipeline Steps:
1. Data Loading
2. Exploratory Data Analysis (EDA)
3. Data Preprocessing
4. Feature Engineering
5. Model Training and Comparison
6. Model Evaluation
7. Model Selection



In [1]:
# Import necessary libraries
import sys
import os
sys.path.append(os.path.join(os.path.dirname(os.getcwd()), 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Import custom modules
from data_loader import load_dataset, get_column_info
from preprocessing import DataPreprocessor
from model_trainer import ModelTrainer
from utils import display_data_info, plot_confusion_matrix, plot_feature_importance

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("Libraries imported successfully!")



Libraries imported successfully!


## 1. Data Loading

Load the cybersecurity attacks dataset from the data folder.


In [3]:
# Load the dataset
try:
    df = load_dataset()
    print(f"\nDataset shape: {df.shape}")
    print(f"\nFirst few rows:")
    df.head()
except FileNotFoundError as e:
    print(f"Error: {e}")
    print("\nPlease place your dataset file (CSV or Excel) in the 'data/' folder.")
    print("Expected file names: cybersecurity_attacks.csv, cybersecurity_attacks.xlsx, etc.")



Error: Dataset file not found. Please place your CSV/Excel file in the 'data/' folder.
Available files in data/: []

Please place your dataset file (CSV or Excel) in the 'data/' folder.
Expected file names: cybersecurity_attacks.csv, cybersecurity_attacks.xlsx, etc.


In [4]:
# Check column information
if 'df' in locals():
    col_info = get_column_info(df)
    print("Expected columns:", len(col_info['expected']))
    print("Actual columns:", len(col_info['actual']))
    print("\nColumn names in dataset:")
    for i, col in enumerate(df.columns, 1):
        print(f"  {i:2d}. {col}")
    
    if col_info['missing']:
        print(f"\n⚠️  Missing expected columns: {col_info['missing']}")
    if col_info['extra']:
        print(f"\nℹ️  Extra columns found: {col_info['extra']}")



## 2. Exploratory Data Analysis (EDA)

Perform comprehensive exploratory data analysis to understand the dataset.


In [5]:
# Display comprehensive dataset information
if 'df' in locals():
    display_data_info(df)



In [6]:
# Visualize target variable distribution
if 'df' in locals() and 'Attack Type' in df.columns:
    plt.figure(figsize=(12, 6))
    attack_counts = df['Attack Type'].value_counts()
    plt.bar(range(len(attack_counts)), attack_counts.values)
    plt.xticks(range(len(attack_counts)), attack_counts.index, rotation=45, ha='right')
    plt.xlabel('Attack Type')
    plt.ylabel('Frequency')
    plt.title('Distribution of Attack Types')
    plt.tight_layout()
    plt.show()
    
    print(f"\nAttack Type Distribution:")
    print(attack_counts)
    print(f"\nTotal unique attack types: {df['Attack Type'].nunique()}")



In [11]:
# Check for missing values visualization
if 'df' in locals():
    missing_data = df.isnull().sum()
    missing_data = missing_data[missing_data > 0].sort_values(ascending=False)
    
    if len(missing_data) > 0:
        plt.figure(figsize=(12, 6))
        plt.barh(range(len(missing_data)), missing_data.values)
        plt.yticks(range(len(missing_data)), missing_data.index)
        plt.xlabel('Number of Missing Values')
        plt.title('Missing Values by Column')
        plt.tight_layout()
        plt.show()
        
        print("\nMissing values summary:")
        print(missing_data)
    else:
        print("✓ No missing values found in the dataset!")



In [12]:
# Correlation analysis for numerical features
if 'df' in locals():
    numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if 'Attack Type' in numerical_cols:
        numerical_cols.remove('Attack Type')
    
    if len(numerical_cols) > 1:
        # Encode target for correlation if it's categorical
        if df['Attack Type'].dtype == 'object':
            from sklearn.preprocessing import LabelEncoder
            le = LabelEncoder()
            df_temp = df.copy()
            df_temp['Attack Type_encoded'] = le.fit_transform(df_temp['Attack Type'])
            corr_cols = numerical_cols + ['Attack Type_encoded']
        else:
            corr_cols = numerical_cols + ['Attack Type']
        
        correlation_matrix = df_temp[corr_cols].corr() if 'df_temp' in locals() else df[corr_cols].corr()
        
        plt.figure(figsize=(14, 12))
        sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
                   square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
        plt.title('Correlation Matrix of Numerical Features')
        plt.tight_layout()
        plt.show()



## 3. Data Preprocessing

Clean and preprocess the data for machine learning.


In [ ]:
# Initialize preprocessor
preprocessor = DataPreprocessor()

# Preprocess the data
if 'df' in locals() and 'Attack Type' in df.columns:
    print("Preprocessing data...")
    X, y = preprocessor.preprocess(df, target_col='Attack Type', fit=True)
    
    print(f"\nPreprocessed data shape: X={X.shape}, y={y.shape}")
    print(f"Number of features: {X.shape[1]}")
    print(f"Number of classes: {len(np.unique(y))}")
    print(f"\nClass distribution (encoded):")
    unique, counts = np.unique(y, return_counts=True)
    for u, c in zip(unique, counts):
        print(f"  Class {u}: {c} samples ({c/len(y)*100:.2f}%)")
    
    # Save preprocessor
    os.makedirs('../models', exist_ok=True)
    preprocessor.save('../models/preprocessor.pkl')
    print("\n✓ Preprocessor saved to models/preprocessor.pkl")
else:
    print("⚠️  Dataset not loaded or 'Attack Type' column not found!")



In [ ]:
# Split data into train and validation sets
from sklearn.model_selection import train_test_split

if 'X' in locals() and 'y' in locals():
    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    
    print(f"Training set: {X_train.shape[0]} samples")
    print(f"Validation set: {X_val.shape[0]} samples")
    print(f"\nTraining set class distribution:")
    unique, counts = np.unique(y_train, return_counts=True)
    for u, c in zip(unique, counts):
        print(f"  Class {u}: {c} samples")
else:
    print("⚠️  Preprocessed data not available!")



## 4. Model Training and Comparison

Train multiple models and compare their performance.


In [ ]:
# Initialize and train models
if 'X_train' in locals() and 'y_train' in locals():
    trainer = ModelTrainer()
    trainer.train_models(X_train, y_train, X_val, y_val)
    
    # Display results summary
    print("\n" + "="*80)
    print("MODEL COMPARISON RESULTS")
    print("="*80)
    results_df = trainer.get_results_summary()
    print(results_df.to_string(index=False))
else:
    print("⚠️  Training data not available!")



## 5. Model Evaluation

Evaluate the best model with detailed metrics.


In [ ]:
# Get best model and evaluate
if 'trainer' in locals():
    best_model, best_model_name = trainer.get_best_model()
    
    # Get predictions from best model
    y_pred = trainer.results[best_model_name]['predictions']
    
    # Classification report
    from sklearn.metrics import classification_report
    print(f"\n{'='*80}")
    print(f"CLASSIFICATION REPORT - {best_model_name}")
    print(f"{'='*80}\n")
    
    # Get class names
    class_names = preprocessor.target_encoder.classes_
    print(classification_report(y_val, y_pred, target_names=class_names))
    
    # Confusion Matrix
    plt.figure(figsize=(12, 10))
    cm = plot_confusion_matrix(y_val, y_pred, class_names, 
                              title=f'Confusion Matrix - {best_model_name}')
    plt.show()
else:
    print("⚠️  Model trainer not available!")



In [ ]:
# Feature importance visualization (for tree-based models)
if 'trainer' in locals() and 'preprocessor' in locals():
    best_model, best_model_name = trainer.get_best_model()
    
    if hasattr(best_model, 'feature_importances_'):
        feature_names = preprocessor.feature_columns
        fig = plot_feature_importance(best_model, feature_names, top_n=20)
        if fig:
            plt.show()
    else:
        print(f"{best_model_name} does not support feature importance visualization.")



## 6. Save Best Model

Save the best performing model for deployment.


In [13]:
# Save the best model
if 'trainer' in locals():
    best_model, best_model_name = trainer.get_best_model()
    
    model_path = '../models/best_model.pkl'
    trainer.save_best_model(model_path)
    
    print(f"\n✓ Best model ({best_model_name}) saved successfully!")
    print(f"  Model path: {model_path}")
    print(f"  Preprocessor path: ../models/preprocessor.pkl")
    
    # Save results summary
    results_df = trainer.get_results_summary()
    results_df.to_csv('../models/model_comparison.csv', index=False)
    print(f"  Results saved to: ../models/model_comparison.csv")
else:
    print("⚠️  Model trainer not available!")



⚠️  Model trainer not available!
